# Train Rhino LLM (QLoRA on Colab T4)

Fine-tune **Qwen2.5-Coder-3B-Instruct** on `rhino_sft.jsonl` with [Unsloth](https://github.com/unslothai/unsloth).

**Runtime:** GPU (T4). Enable Drive for checkpoints.

In [ ]:
!nvidia-smi

In [ ]:
# Install Unsloth + training stack (restart runtime if Colab asks)
%%capture
import os, subprocess
os.environ["UNSLOTH_IS_COLAB"] = "1"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade transformers datasets trl peft accelerate bitsandbytes pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Hugging Face token (hf_...): ")
os.environ["HF_TOKEN"] = HF_TOKEN
!huggingface-cli login --token $HF_TOKEN

# Cache models on Drive (optional)
HF_HOME = "/content/drive/MyDrive/hf_cache"
os.makedirs(HF_HOME, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME

In [ ]:
# Clone repo OR upload rhino_sft.jsonl to Drive
import os
from pathlib import Path

REPO_DIR = Path("/content/My-LLM")
if not REPO_DIR.exists():
    !git clone https://github.com/YOUR_USER/My-LLM.git /content/My-LLM

DATA_PATH = REPO_DIR / "data/processed/rhino_sft.jsonl"
DRIVE_DATA = "/content/drive/MyDrive/rhino_llm/rhino_sft.jsonl"
if Path(DRIVE_DATA).exists():
    DATA_PATH = Path(DRIVE_DATA)

assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"
print("Dataset:", DATA_PATH)

In [ ]:
import yaml
from pathlib import Path

cfg_path = REPO_DIR / "configs/train_qlora.yaml"
if not cfg_path.exists():
    cfg_path = Path("configs/train_qlora.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

MODEL_NAME = cfg["model"]["base"]
FALLBACK_MODEL = cfg["model"]["fallback_base"]
MAX_SEQ_LENGTH = cfg["training"]["max_seq_length"]
SYSTEM_PROMPT = cfg["system_prompt"].strip()
OUTPUT_DIR = "/content/drive/MyDrive/rhino_llm/" + cfg["export"]["adapter_dir"]

USE_FALLBACK = False  # set True if OOM
if USE_FALLBACK:
    MODEL_NAME = FALLBACK_MODEL
    MAX_SEQ_LENGTH = 1024

print("Model:", MODEL_NAME, "| max_seq:", MAX_SEQ_LENGTH)

In [ ]:
import json

rows = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples")

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora"]["r"],
    lora_alpha=cfg["lora"]["lora_alpha"],
    lora_dropout=cfg["lora"]["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias=cfg["lora"]["bias"],
    use_gradient_checkpointing="unsloth",
    random_state=cfg["data"]["seed"],
)

from datasets import Dataset

def to_text(example):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + example["messages"]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(rows)
ds = ds.map(to_text)
print(ds[0]["text"][:400], "...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

t = cfg["training"]
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=cfg["training"]["packing"],
    args=TrainingArguments(
        per_device_train_batch_size=t["per_device_train_batch_size"],
        gradient_accumulation_steps=t["gradient_accumulation_steps"],
        warmup_ratio=t["warmup_ratio"],
        num_train_epochs=t["num_train_epochs"],
        learning_rate=t["learning_rate"],
        logging_steps=t["logging_steps"],
        optim=t["optim"],
        weight_decay=t["weight_decay"],
        lr_scheduler_type=t["lr_scheduler_type"],
        seed=cfg["data"]["seed"],
        output_dir="/content/rhino_checkpoints",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer.train()

In [ ]:
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)

## Eval (before/after)

Runs 5 held-out prompts from `eval/prompts.jsonl` and optional heuristic scoring.

In [ ]:
import json
from pathlib import Path
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

eval_path = REPO_DIR / "eval/prompts.jsonl"
prompts = []
with open(eval_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        if line.strip():
            prompts.append(json.loads(line))

def generate(prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.2)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

predictions = []
for p in prompts:
    out = generate(p["prompt"])
    predictions.append({
        "id": p["id"],
        "prompt": p["prompt"],
        "output": out,
        "meta": p.get("meta", {}),
    })
    print("=" * 60)
    print(p["id"], "|", p["meta"].get("lang"))
    print(out[:800])

In [ ]:
# Heuristic scoring (run locally or in Colab)
import subprocess, json
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/rhino_llm/eval_runs")
run_dir.mkdir(parents=True, exist_ok=True)
pred_path = run_dir / "post_train.jsonl"
with pred_path.open("w", encoding="utf-8") as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

scorer = REPO_DIR / "eval/score_outputs.py"
if scorer.exists():
    !python {scorer} --predictions {pred_path}
else:
    print("Saved predictions to", pred_path)

## OOM fallbacks

If training crashes with CUDA OOM:
1. Restart runtime
2. Set `USE_FALLBACK = True` (1.5B model, seq 1024)
3. Reduce LoRA rank to 16 in config or above
4. Set `gradient_accumulation_steps` to 16, batch size 1

**Alternative without Unsloth:** use `peft` + `bitsandbytes` with `AutoModelForCausalLM.from_pretrained(..., load_in_4bit=True)` and standard `SFTTrainer` — slower but works if Unsloth install fails.